# 05 · Extension: recognise the exact car model (196 classes)

In [ ]:
import os, sys
REPO_URL = "https://github.com/YUPWE/car-size.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/car-size"
    if not os.path.exists(ROOT):
        !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
    !git pull -q
    !pip install -q imagehash==4.3.2
    os.environ["CARS_DATA_DIR"] = "/content/cars_data"
else:
    ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.makedirs("results", exist_ok=True)
os.makedirs("weights", exist_ok=True)
QUICK = os.environ.get("QUICK", "0") == "1"
print("Working in", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
import json, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score
from src import data
from src.data import CarDataset, get_transforms
from src.model import build_resnet18, train, predict_probs, set_seed, get_device
from src.clip_gate import ClipZeroShot, load_pil

data.download()
device = get_device()
df = data.load_meta()
names = df.drop_duplicates("model_id").sort_values("model_id").model.tolist()
print(len(df), "images,", len(names), "car models,", device)

## Split: every model in train, val and test; duplicates kept together

In [ ]:
short = df[["orig_w", "orig_h"]].min(axis=1)
hashes = data.perceptual_hashes(df.path)
pairs = data.near_duplicate_pairs(hashes, max_dist=4)
conflict = {k for i, j in pairs if df.model[i] != df.model[j] for k in (i, j)}
drop = sorted(set(df.index[short < 64]) | conflict)
clean = df.drop(index=drop).reset_index(drop=True)
keep = {old: new for new, old in enumerate(df.index.difference(drop))}
clean_pairs = [(keep[i], keep[j]) for i, j in pairs if i in keep and j in keep]
groups = data.duplicate_groups(len(clean), clean_pairs)
print(f"dropped {len(drop)} images; {len(clean_pairs)} duplicate pairs kept inside one split")

split = data.make_model_splits(clean, groups, seed=42)
print(split.split.value_counts())
per_model = split.groupby(["model_id", "split"]).size().unstack(fill_value=0)
print("models missing from a split:", int((per_model == 0).any(axis=1).sum()))
print("test images per model: min", per_model.test.min(), "| median", int(per_model.test.median()))
g = pd.Series(groups)
assert split.groupby(g).split.nunique().max() == 1
for s in ["train", "val", "test"]:
    split[split.split == s][["path", "model_id", "model", "label"]].to_csv(f"splits/models_{s}.csv", index=False)
json.dump(names, open("results/car_model_names.json", "w"), indent=0)

## Train ResNet-18 with 196 outputs

In [ ]:
QE = 1 if QUICK else 10
tr, va, te = (pd.read_csv(f"splits/models_{s}.csv") for s in ["train", "val", "test"])
if QUICK:
    tr, va, te = tr.sample(128, random_state=0), va.sample(64, random_state=0), te.sample(64, random_state=0)
mk = lambda d, t: CarDataset(d, get_transforms(t), targets=d.model_id)
train_loader = DataLoader(mk(tr, True), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(mk(va, False), batch_size=128, num_workers=2)
test_loader = DataLoader(mk(te, False), batch_size=128, num_workers=2)

set_seed(42)
net = build_resnet18(freeze_backbone=False, num_classes=len(names))
net, hist = train(net, train_loader, val_loader, epochs=QE, lr=5e-4, device=device)
torch.save(net.state_dict(), "weights/resnet18_carmodel.pt")
json.dump(hist, open("results/car_model_history.json", "w"), indent=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ep = range(1, len(hist["train_loss"]) + 1)
ax[0].plot(ep, hist["train_loss"], label="train"); ax[0].plot(ep, hist["val_loss"], label="val"); ax[0].set_title("Loss")
ax[1].plot(ep, hist["train_acc"], label="train"); ax[1].plot(ep, hist["val_acc"], label="val"); ax[1].set_title("Accuracy (196 models)")
for a in ax: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/car_model_curves.png", dpi=120); plt.show()

## CLIP zero-shot: read the 196 names, no training

In [ ]:
clip = ClipZeroShot(device=device)
T = clip.text_embeddings([f"a photo of a {n}, a type of car." for n in names])
scale = clip.model.logit_scale.exp().item()

def clip_probs(d):
    E = clip.image_embeddings(load_pil(f"{data.IMG_DIR}/{p}" for p in d.path))
    z = scale * E @ T.T
    z = np.exp(z - z.max(1, keepdims=True))
    return z / z.sum(1, keepdims=True)

pv_res, yv = predict_probs(net, val_loader, device)
pt_res, yt = predict_probs(net, test_loader, device)
pv_clip, pt_clip = clip_probs(va), clip_probs(te)

ws = np.round(np.linspace(0, 1, 11), 1)
val_acc = [accuracy_score(yv, (w * pv_res + (1 - w) * pv_clip).argmax(1)) for w in ws]
W5 = float(ws[int(np.argmax(val_acc))])
print("mixing weight chosen on validation:", W5)

## Test results

In [ ]:
def topk(p, y, k):
    return float(np.mean([y[i] in np.argsort(-p[i])[:k] for i in range(len(y))]))

pt_mix = W5 * pt_res + (1 - W5) * pt_clip
rows = []
for name, p in [("CLIP zero-shot (no training)", pt_clip), ("ResNet-18 fine-tuned", pt_res), (f"ResNet + CLIP mix (w={W5})", pt_mix)]:
    rows.append({"model": name, "top-1 accuracy": topk(p, yt, 1), "top-5 accuracy": topk(p, yt, 5),
                 "macro-F1": f1_score(yt, p.argmax(1), average="macro")})
table = pd.DataFrame(rows).round(3)
print(f"Test set: {len(yt)} images, {len(set(yt))} car models")
print(table.to_markdown(index=False))
table.to_csv("results/car_model_comparison.csv", index=False)

In [ ]:
pred = pt_mix.argmax(1)
conf = pd.DataFrame({"true": [names[i] for i in yt], "predicted": [names[i] for i in pred]})
conf = conf[conf.true != conf.predicted].value_counts().reset_index(name="count").head(15)
print(conf.to_string(index=False))
conf.to_csv("results/car_model_confusions.csv", index=False)

te2 = te.reset_index(drop=True)
idx = np.random.default_rng(0).choice(len(te2), size=min(12, len(te2)), replace=False)
fig, axes = plt.subplots(2, 6, figsize=(20, 7)); axes = axes.ravel()
for a, i in zip(axes, idx):
    a.imshow(plt.imread(f"{data.IMG_DIR}/{te2.path[i]}")); a.axis("off")
    ok = pred[i] == yt[i]
    a.set_title(f"{'OK' if ok else 'WRONG'}: {names[pred[i]][:28]}\ntrue: {names[yt[i]][:28]}", fontsize=8,
                color="green" if ok else "red")
plt.tight_layout(); plt.savefig("results/car_model_examples.png", dpi=100); plt.show()